# Sentiment Analysis on Social Media Data
### Final Project Submission — Project Acharya Internship

This notebook shows my end-to-end sentiment analysis pipeline. It covers:
* Preprocessing text (cleaning up URLs, handles, and lemmatizing)
* Feature extraction (using TF-IDF for linear models and padding sequences for deep learning)
* Model training (comparing Naive Bayes, Logistic Regression, and a Bidirectional LSTM)
* Plotting metrics and evaluating results

## 1. importing needed libraries


In [ ]:
import json
import os
import pickle
import sys

import joblib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    LSTM,
    BatchNormalization,
    Bidirectional,
    Dense,
    Dropout,
    Embedding,
    GlobalMaxPooling1D,
    SpatialDropout1D,
)
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from wordcloud import WordCloud

print("TensorFlow Version:", tf.__version__)
print("NLTK Version:", nltk.__version__)

## 2. Loading and Traversing the Dataset
We load the raw Sentiment140 dataset, map target labels (0 for -ve, 1 for +ve), and take a balanced sample of 100k rows.

In [ ]:
# load sentiment140 dataset from the data raw directory
raw_path = "../data/raw/sentiment140.csv"
cols = ["target", "ids", "date", "flag", "user", "text"]
# the encoding has to be iso-8859-1 for this csv, otherwise pandas will throw an error
df = pd.read_csv(raw_path, encoding="ISO-8859-1", header=None, names=cols)
# map labels: 0 remains 0 (negative), 4 becomes 1 (positive)
df["target"] = df["target"].map({0: 0, 4: 1})
# drop null target records if any exist
df = df.dropna(subset=["target"])
# take a smaller balanced sample to speed up training in the notebook
df_pos = df[df["target"] == 1].sample(n=5000, random_state=42)
df_neg = df[df["target"] == 0].sample(n=5000, random_state=42)
df = pd.concat([df_pos, df_neg]).sample(frac=1.0, random_state=42).reset_index(drop=True)
df.head()

### distribution of the sentiment

In [ ]:
# let us plot a countplot to confirm classes are perfectly balanced (50/50 ratio)
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="target", palette=["#FF6B6B", "#4ECDC4"])
plt.title("class distribution in our sample")
plt.show()

## 3. Text Preprocessing
cleaning up the tweets by stripping URLs, handles, and special characters, expanding common contractions, and removing stopwords before lemmatizing it.

In [ ]:
# download nltk dependencies for text processing
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

# text cleaning function to strip urls, user handles, etc.
def preprocess_tweet(text):
    if not isinstance(text, str):
        return ""
    # lowercase everything first
    text = text.lower()
    # replace urls, mentions, rt indicators, etc.
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\brt\b", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    # normalize extra spaces
    text = " ".join(text.split()).strip()
    
    # tokenize and remove stopwords/lemmatize tokens
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    return " ".join(tokens)

# apply preprocessing to the entire dataframe
df["processed_text"] = df["text"].apply(preprocess_tweet)
# drop empty rows in case some tweets become empty after cleaning
df = df[df["processed_text"].str.len() > 0].reset_index(drop=True)
df[["text", "processed_text"]].head()

## 4. Feature Engineering
We split the dataset and convert the preprocessed texts into numeric vectors. into features:
* TF-IDF vectors for Naive Bayes and Logistic Regression
* Padded integer sequences for the LSTM

In [ ]:
X = df["processed_text"].fillna("")
y = df["target"]

# split dataset into stratified train and test sets (20% for testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train):,}")
print(f"Testing set: {len(X_test):,}")

### 4.1 TF-IDF Vectorization

In [ ]:
# fit tf-idf vectorizer on training text features
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=5, max_df=0.95)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF training matrix shape:", X_train_tfidf.shape)

### 4.2 Tokenization and Padded Sequences for LSTM

In [ ]:
# set maximum vocabulary and max sequence length parameters for padding
max_vocab = 20000
max_length = 100

# fit keras tokenizer on the training texts
tokenizer = Tokenizer(num_words=max_vocab, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

# transform and pad sequences to uniform length of 100
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_length, padding="post", truncating="post")
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_length, padding="post", truncating="post")

print("LSTM padded sequence shape:", X_train_seq.shape)

## 5. Model Training and Evaluation

### 5.1 Logistic Regression


In [ ]:
# initialize and fit logistic regression classifier
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

# run predictions and calculate confidence scores (probabilities)
lr_preds = lr_model.predict(X_test_tfidf)
lr_probas = lr_model.predict_proba(X_test_tfidf)[:, 1]

print("--- Logistic Regression Results ---")
print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print(classification_report(y_test, lr_preds, target_names=["Negative", "Positive"]))

### 5.2 Naive Bayes Classifier


In [ ]:
# initialize and fit multinomial naive bayes classifier as baseline
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_tfidf, y_train)

# predict labels and probabilities on test set
nb_preds = nb_model.predict(X_test_tfidf)
nb_probas = nb_model.predict_proba(X_test_tfidf)[:, 1]

print("--- Naive Bayes Results ---")
print(f"Accuracy: {accuracy_score(y_test, nb_preds):.4f}")
print(classification_report(y_test, nb_preds, target_names=["Negative", "Positive"]))

### 5.3 Bidirectional LSTM

In [ ]:
# build sequential bidirectional lstm using keras embedding layer
vocab_size = min(len(tokenizer.word_index) + 1, max_vocab)

lstm_model = Sequential([
    Embedding(vocab_size, 64, input_length=max_length),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    BatchNormalization(),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

# compile model using adam with binary crossentropy loss
lstm_model.compile(optimizer=Adam(learning_rate=0.002), loss="binary_crossentropy", metrics=["accuracy"])
lstm_model.summary()

In [ ]:
# set up early stopping to monitor val loss
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

# train the lstm model for 5 epochs
history = lstm_model.fit(
    X_train_seq, y_train.values,
    validation_split=0.1,
    epochs=5, batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# run test predictions and cast probability to binary outputs
lstm_probas = lstm_model.predict(X_test_seq).flatten()
lstm_preds = (lstm_probas >= 0.5).astype(int)

print("--- LSTM Results ---")
print(f"Accuracy: {accuracy_score(y_test, lstm_preds):.4f}")
print(classification_report(y_test, lstm_preds, target_names=["Negative", "Positive"]))

## 6. Visualizing the model


### 6.1 Model Performance Comparison

In [ ]:
# draw comparative bar chart for accuracy and f1-score
models = ["Naive Bayes", "Logistic Regression", "LSTM"]
accuracies = [
    accuracy_score(y_test, nb_preds),
    accuracy_score(y_test, lr_preds),
    accuracy_score(y_test, lstm_preds)
]
f1_scores = [
    f1_score(y_test, nb_preds, average="weighted"),
    f1_score(y_test, lr_preds, average="weighted"),
    f1_score(y_test, lstm_preds, average="weighted")
]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, accuracies, width, label="Accuracy", color="#667eea")
ax.bar(x + width/2, f1_scores, width, label="F1-Score (Weighted)", color="#764ba2")

ax.set_ylabel("Score")
ax.set_title("Performance Comparison")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0.0, 1.0)
ax.legend()
plt.show()

### 6.2 Confusion Matrices

In [ ]:
# draw confusion matrices side-by-side to compare incorrect prediction ratios
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cms = [
    (confusion_matrix(y_test, nb_preds), "Naive Bayes"),
    (confusion_matrix(y_test, lr_preds), "Logistic Regression"),
    (confusion_matrix(y_test, lstm_preds), "LSTM")
]

for ax, (matrix, name) in zip(axes, cms):
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices comparison")
plt.show()

### 6.3 ROC Curves

In [ ]:
# compute false positive / true positive rate arrays and plot roc curves
plt.figure(figsize=(8, 6))

for probas, name in [(nb_probas, "Naive Bayes"), (lr_probas, "Logistic Regression"), (lstm_probas, "LSTM")]:
    fpr, tpr, _ = roc_curve(y_test, probas)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.4f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend(loc="lower right")
plt.show()

### 6.4 Word Clouds by Sentiment

In [ ]:
# build positive and negative word clouds to visualize common keywords
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, sentiment, title, cmap in zip(
    axes, [0, 1], ["Negative Sentiment", "Positive Sentiment"], ["Reds", "Greens"]
):
    text = " ".join(df[df["target"] == sentiment]["processed_text"].dropna().values)
    wc = WordCloud(width=800, height=400, max_words=100, background_color="white", colormap=cmap).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(title, fontsize=14, fontweight="bold")

plt.show()

### 6.5 Top Predictors

In [ ]:
# extract regression coefficients and plot terms with most positive/negative weights
feature_names = vectorizer.get_feature_names_out()
coefs = lr_model.coef_[0]
top_pos_idx = np.argsort(coefs)[-15:]
top_neg_idx = np.argsort(coefs)[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(range(15), coefs[top_pos_idx], color="#4ECDC4")
axes[0].set_yticks(range(15))
axes[0].set_yticklabels(feature_names[top_pos_idx])
axes[0].set_title("Top Positive Predictors")

axes[1].barh(range(15), coefs[top_neg_idx], color="#FF6B6B")
axes[1].set_yticks(range(15))
axes[1].set_yticklabels(feature_names[top_neg_idx])
axes[1].set_title("Top Negative Predictors")

plt.suptitle("Word Coefficients in Logistic Regression")
plt.tight_layout()
plt.show()

## 7. Conclusion

We built and compared three model setups for sentiment classification:
* **Logistic Regression** achieved the highest accuracy (**76.17%**) and ROC-AUC (**0.8418**).
* **Bidirectional LSTM** trained quickly and scored **75.33%** accuracy.
* **Naive Bayes** serves as a baseline at **74.94%** accuracy.

All models and tokenizers are saved and ready for deployment in the Streamlit app.